Lab09

Bai01

In [1]:
# -*- coding: utf-8 -*-
"""Buoc 1: tu tinh lan truyen tien bang numpy.

Chu thich viet tieng Viet co dau, phan in ra man hinh viet khong dau.
"""
import numpy as np


def sigmoid(z):
    return 1 / (1 + np.exp(-z))


def relu(z):
    return np.maximum(0, z)


# Phan 1: mot no-ron voi ham sigmoid, chinh la hoi quy logistic
w = np.array([0.8, -0.5, 0.3])
b = -0.2
x = np.array([2.0, 1.0, 3.0])
z = w @ x + b
print("Mot no-ron:")
print(f" z = {w[0]}*{x[0]} + ({w[1]})*{x[1]} + {w[2]}*{x[2]} + ({b}) = {z:.2f}")
print(f" a = sigmoid(z) = {sigmoid(z):.4f}")
print()

# Phan 2: mang 2 dau vao, 2 no-ron an ReLU, 1 no-ron ra
# Trong so tu dat de mang tinh dung phep XOR
W1 = np.array([
    [1.0, 1.0],   # cot 1: trong so vao no-ron an h1
    [1.0, 1.0],   # cot 2: trong so vao no-ron an h2
])
b1 = np.array([0.0, -1.0])
W2 = np.array([1.0, -2.0])
b2 = 0.0

X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
print("Mang giai XOR:")
print(" x1 x2 h1 h2 y")
for x in X:
    h = relu(x @ W1 + b1)     # Lop an
    y = h @ W2 + b2           # Lop ra
    print(f" {x[0]:.0f} {x[1]:.0f} {h[0]:.0f} {h[1]:.0f} {y:.0f}")
print()

# Ca bon dau vao cung luc bang phep nhan ma tran
H = relu(X @ W1 + b1)
Y = H @ W2 + b2
print("Tinh ca bang mot lan:", Y)

Mot no-ron:
 z = 0.8*2.0 + (-0.5)*1.0 + 0.3*3.0 + (-0.2) = 1.80
 a = sigmoid(z) = 0.8581

Mang giai XOR:
 x1 x2 h1 h2 y
 0 0 0 0 0
 0 1 1 0 1
 1 0 1 0 1
 1 1 2 1 0

Tinh ca bang mot lan: [0. 1. 1. 0.]


Bai02

In [2]:
# -*- coding: utf-8 -*-
"""Buoc 2: mot no-ron khong giai duoc XOR, mot lop an thi duoc."""
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier

df = pd.read_csv("xor.csv")
X = df[["x1", "x2"]]
y = df["nhan"]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

lr = LogisticRegression().fit(X_tr, y_tr)
print(f"Hoi quy logistic: {accuracy_score(y_te, lr.predict(X_te)):.4f}")
print()

# Du lieu nho nen dung solver="lbfgs", hoc nhanh va on dinh hon
print("So no-ron an  so tham so  tap hoc  tap kiem tra")
for so in [1, 2, 4, 8]:
    mlp = MLPClassifier(hidden_layer_sizes=(so,), solver="lbfgs",
                        max_iter=5000, random_state=0)
    mlp.fit(X_tr, y_tr)
    # coefs_ la danh sach ma tran trong so, intercepts_ la he so chan
    so_ts = sum(w.size for w in mlp.coefs_)
    so_ts += sum(b.size for b in mlp.intercepts_)
    a_hoc = accuracy_score(y_tr, mlp.predict(X_tr))
    a_kt = accuracy_score(y_te, mlp.predict(X_te))
    print(f" {so:2d}          {so_ts:3d}        {a_hoc:.4f}   {a_kt:.4f}")

Hoi quy logistic: 0.4500

So no-ron an  so tham so  tap hoc  tap kiem tra
  1            5        0.6533   0.6900
  2            9        0.6533   0.6900
  4           17        0.9033   0.9300
  8           33        0.9167   0.9100


bai03

In [3]:
# -*- coding: utf-8 -*-
"""Buoc 3: do lon mang va dieu chuan (alpha) tren XOR."""
import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("xor.csv")
X = df[["x1", "x2"]]
y = df["nhan"]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

print("=== 4.1 Bang do lon mang x alpha, cham tren tap kiem tra ===")
print("So no-ron an |  alpha  |  tap hoc  |  tap kiem tra")
for so in [2, 4, 8, 16, 32]:
    for alpha in [0.0001, 0.001, 0.01, 0.1]:
        mlp = MLPClassifier(hidden_layer_sizes=(so,), solver="lbfgs",
                            alpha=alpha, max_iter=5000, random_state=0)
        mlp.fit(X_tr, y_tr)
        a_hoc = accuracy_score(y_tr, mlp.predict(X_tr))
        a_kt = accuracy_score(y_te, mlp.predict(X_te))
        print(f" {so:>2d}         | {alpha:7.4f} |  {a_hoc:.4f}   |  {a_kt:.4f}")
print()

print("=== 4.2 Chon alpha bang kiem dinh cheo tren tap hoc ===")
# Luoi: vua so no-ron an vua alpha, chon bang 5-fold CV tren tap hoc
luoi = {
    "mlpclassifier__hidden_layer_sizes": [(2,), (4,), (8,), (16,), (32,)],
    "mlpclassifier__alpha": [0.0001, 0.001, 0.01, 0.1],
}
ong = make_pipeline(
    StandardScaler(),
    MLPClassifier(solver="lbfgs", max_iter=5000, random_state=0),
)
tim = GridSearchCV(ong, luoi, cv=5, scoring="accuracy", n_jobs=-1)
tim.fit(X_tr, y_tr)
print(f"Bo tham so tot nhat tren tap hoc: {tim.best_params_}")
print(f"Diem kiem dinh cheo tot nhat:      {tim.best_score_:.4f}")

# Chon xong moi mo tap kiem tra, chi mot lan
a_kt = accuracy_score(y_te, tim.predict(X_te))
print(f"Accuracy tren tap kiem tra:        {a_kt:.4f}")

=== 4.1 Bang do lon mang x alpha, cham tren tap kiem tra ===
So no-ron an |  alpha  |  tap hoc  |  tap kiem tra
  2         |  0.0001 |  0.6533   |  0.6900
  2         |  0.0010 |  0.6533   |  0.6800
  2         |  0.0100 |  0.6533   |  0.6800
  2         |  0.1000 |  0.6533   |  0.6800
  4         |  0.0001 |  0.9033   |  0.9300
  4         |  0.0010 |  0.9033   |  0.9300
  4         |  0.0100 |  0.9067   |  0.9200
  4         |  0.1000 |  0.9067   |  0.9300
  8         |  0.0001 |  0.9167   |  0.9100
  8         |  0.0010 |  0.9167   |  0.9200
  8         |  0.0100 |  0.9033   |  0.9200
  8         |  0.1000 |  0.9200   |  0.9300
 16         |  0.0001 |  0.9267   |  0.9300
 16         |  0.0010 |  0.9200   |  0.9200
 16         |  0.0100 |  0.9133   |  0.9300
 16         |  0.1000 |  0.9200   |  0.9400
 32         |  0.0001 |  0.9500   |  0.9400
 32         |  0.0010 |  0.9533   |  0.9400
 32         |  0.0100 |  0.9133   |  0.9200
 32         |  0.1000 |  0.9200   |  0.9300

=== 4.2

Bai04

In [4]:
# -*- coding: utf-8 -*-
"""Buoc 4: mang no-ron bat buoc phai chuan hoa.

Demo tren sinh_vien_mo_rong.csv: neu quen chuan hoa, solver adam/lbfgs
kho hoi tu va ket qua te hon han.
"""
import warnings
import numpy as np
import pandas as pd
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=ConvergenceWarning)

df = pd.read_csv("sinh_vien_mo_rong.csv")
nhan = "bo_hoc" if "bo_hoc" in df.columns else df.columns[-1]
X = df.drop(columns=nhan)
y = df[nhan]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

# Xem qua do lon cac cot de thay vi sao phai chuan hoa
print("Do lech chuan tung cot tren tap hoc:")
print(X_tr.std().round(2).to_string())
print()

cau_hinh = dict(hidden_layer_sizes=(16, 16), max_iter=3000, random_state=0)

print("=== Khong chuan hoa ===")
mlp_tho = MLPClassifier(**cau_hinh).fit(X_tr, y_tr)
du_tho = mlp_tho.predict(X_te)
print(f" So vong da chay: {mlp_tho.n_iter_}")
print(f" Accuracy:        {accuracy_score(y_te, du_tho):.4f}")
print(f" F1 lop bo hoc:   {f1_score(y_te, du_tho):.4f}")
print()

print("=== Co chuan hoa (StandardScaler) ===")
mlp_chuan = make_pipeline(
    StandardScaler(),
    MLPClassifier(**cau_hinh),
).fit(X_tr, y_tr)
du_chuan = mlp_chuan.predict(X_te)
mlp_fit = mlp_chuan.named_steps["mlpclassifier"]
print(f" So vong da chay: {mlp_fit.n_iter_}")
print(f" Accuracy:        {accuracy_score(y_te, du_chuan):.4f}")
print(f" F1 lop bo hoc:   {f1_score(y_te, du_chuan):.4f}")
print()
print(f"So du doan khac nhau giua hai cach: "
      f"{int(np.sum(du_tho != du_chuan))} tren {len(y_te)}")

Do lech chuan tung cot tren tap hoc:
gio_on             8.13
diem_giua_ky       2.16
chi_tieu        1720.71

=== Khong chuan hoa ===
 So vong da chay: 12
 Accuracy:        0.4000
 F1 lop bo hoc:   0.0000

=== Co chuan hoa (StandardScaler) ===
 So vong da chay: 1726
 Accuracy:        0.9000
 F1 lop bo hoc:   0.9143

So du doan khac nhau giua hai cach: 17 tren 30


Bai05

In [5]:
# -*- coding: utf-8 -*-
"""Buoc 5: MLP nhan dang chu so viet tay tren load_digits."""
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

digits = load_digits()
X = digits.data
y = digits.target
print("Kich thuoc du lieu:", X.shape)   # (1797, 64) - 8x8 pixel
print("So lop:", len(np.unique(y)))
print()

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

# 6.1 So tham so
mlp = make_pipeline(
    StandardScaler(),
    MLPClassifier(hidden_layer_sizes=(64,), solver="adam",
                  max_iter=500, random_state=0),
)
mlp.fit(X_tr, y_tr)
mang = mlp.named_steps["mlpclassifier"]

so_ts = sum(w.size for w in mang.coefs_) + sum(b.size for b in mang.intercepts_)
print("Kien truc: 64 -> 64 -> 10")
print(f" Lop an:  64 x 64 trong so + 64 he so chan = {64*64 + 64}")
print(f" Lop ra:  64 x 10 trong so + 10 he so chan = {64*10 + 10}")
print(f" Tong so tham so: {so_ts}")
print()

# 6.2 Duong mat mat
mang_loss = mang.loss_curve_
print(f" So vong da chay: {mang.n_iter_}")
print(f" Mat mat cuoi:    {mang_loss[-1]:.4f}")

plt.figure(figsize=(6, 4))
plt.plot(mang_loss, linewidth=1.5)
plt.xlabel("Vong lap")
plt.ylabel("Loss (cross-entropy)")
plt.title("Duong mat mat cua MLP tren chu so viet tay")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("duong_mat_mat_chu_so.png", dpi=120)
print(" Da luu hinh: duong_mat_mat_chu_so.png")
print()

# 6.3 Ket qua
du = mlp.predict(X_te)
print(f"Accuracy tren tap kiem tra: {accuracy_score(y_te, du):.4f}")
print()
print("Bao cao chi tiet tung lop chu so:")
print(classification_report(y_te, du, digits=4))

Kich thuoc du lieu: (1797, 64)
So lop: 10

Kien truc: 64 -> 64 -> 10
 Lop an:  64 x 64 trong so + 64 he so chan = 4160
 Lop ra:  64 x 10 trong so + 10 he so chan = 650
 Tong so tham so: 4810

 So vong da chay: 168
 Mat mat cuoi:    0.0062
 Da luu hinh: duong_mat_mat_chu_so.png

Accuracy tren tap kiem tra: 0.9756

Bao cao chi tiet tung lop chu so:
              precision    recall  f1-score   support

           0     1.0000    1.0000    1.0000        45
           1     0.9783    0.9783    0.9783        46
           2     0.9565    1.0000    0.9778        44
           3     1.0000    0.9783    0.9890        46
           4     0.9773    0.9556    0.9663        45
           5     1.0000    0.9783    0.9890        46
           6     1.0000    1.0000    1.0000        45
           7     0.9167    0.9778    0.9462        45
           8     0.9524    0.9302    0.9412        43
           9     0.9773    0.9556    0.9663        45

    accuracy                         0.9756       450
 

Bai06

In [6]:
# -*- coding: utf-8 -*-
"""Bai tong hop: MLP tren hai vang trang.

- So sanh hoi quy logistic va MLP.
- Chon kien truc va alpha bang kiem dinh cheo tren tap hoc.
- Ve ranh gioi quyet dinh va duong mat mat.
"""
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=ConvergenceWarning)

df = pd.read_csv("hai_vang_trang.csv")
print("Kich thuoc:", df.shape)
X = df[["x1", "x2"]].to_numpy()
y = df["nhan"].to_numpy()

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

# --- 1) Hoi quy logistic (mot no-ron) ---
lr = make_pipeline(StandardScaler(), LogisticRegression())
lr.fit(X_tr, y_tr)
print(f"Hoi quy logistic: {accuracy_score(y_te, lr.predict(X_te)):.4f}")

# --- 2) MLP voi kien truc va alpha chon bang CV ---
luoi = {
    "mlpclassifier__hidden_layer_sizes": [(4,), (8,), (16,), (16, 16)],
    "mlpclassifier__alpha": [0.0001, 0.001, 0.01, 0.1],
}
ong = make_pipeline(
    StandardScaler(),
    MLPClassifier(solver="adam", max_iter=3000, random_state=0),
)
tim = GridSearchCV(ong, luoi, cv=5, scoring="accuracy", n_jobs=-1)
tim.fit(X_tr, y_tr)
print(f"Bo tham so tot nhat: {tim.best_params_}")
print(f"Diem kiem dinh cheo: {tim.best_score_:.4f}")
print(f"Accuracy tap kiem tra: {accuracy_score(y_te, tim.predict(X_te)):.4f}")
print()

# --- 3) Ve ranh gioi quyet dinh ---
def ve_bien(ax, mo_hinh, tieu_de):
    x_min, x_max = X[:, 0].min() - 0.1, X[:, 0].max() + 0.1
    y_min, y_max = X[:, 1].min() - 0.1, X[:, 1].max() + 0.1
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                         np.linspace(y_min, y_max, 300))
    Z = mo_hinh.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.35, cmap="coolwarm")
    ax.scatter(X[:, 0], X[:, 1], c=y, s=15, cmap="coolwarm", edgecolors="k")
    ax.set_title(tieu_de)
    ax.set_xlabel("x1")
    ax.set_ylabel("x2")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
ve_bien(axes[0], lr, "Hoi quy logistic (mot no-ron)")
ve_bien(axes[1], tim, f"MLP {tim.best_params_['mlpclassifier__hidden_layer_sizes']}")
plt.tight_layout()
plt.savefig("bien_quyet_dinh_hai_vang_trang.png", dpi=120)
print("Da luu hinh: bien_quyet_dinh_hai_vang_trang.png")

# --- 4) Duong mat mat cua mo hinh tot nhat ---
mang = tim.best_estimator_.named_steps["mlpclassifier"]
plt.figure(figsize=(6, 4))
plt.plot(mang.loss_curve_, linewidth=1.5)
plt.xlabel("Vong lap")
plt.ylabel("Loss")
plt.title("Duong mat mat cua MLP tot nhat")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("duong_mat_mat_hai_vang_trang.png", dpi=120)
print("Da luu hinh: duong_mat_mat_hai_vang_trang.png")
print()
so_ts = sum(w.size for w in mang.coefs_) + sum(b.size for b in mang.intercepts_)
print(f"Tong so tham so cua mo hinh tot nhat: {so_ts}")

Kich thuoc: (300, 3)
Hoi quy logistic: 0.8667
Bo tham so tot nhat: {'mlpclassifier__alpha': 0.001, 'mlpclassifier__hidden_layer_sizes': (16, 16)}
Diem kiem dinh cheo: 0.8489
Accuracy tap kiem tra: 0.9067

Da luu hinh: bien_quyet_dinh_hai_vang_trang.png
Da luu hinh: duong_mat_mat_hai_vang_trang.png

Tong so tham so cua mo hinh tot nhat: 337
